# Eksperimen Tahap 6: Evaluasi Hybrid RAG (Vector + Knowledge Graph)

Notebook ini mengevaluasi kinerja metode pencarian secara **Hybrid** (menggabungkan *Vector Retrieval* dan *Graph Retrieval*). 
Sistem akan memecah pencarian ke Jalur A (Vector) dan Jalur B (Graph), lalu menggabungkannya ke dalam satu Prompt (Fusion) sebelum dijawab oleh Mistral AI.

In [ ]:
!pip install sentence-transformers supabase requests pandas evaluate rouge_score nltk

In [ ]:
import os
import requests
import json
import time
import pandas as pd
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer
import evaluate
import nltk
import re

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# --- CONFIGURATION ---
SUPABASE_URL = "https://YOUR_PROJECT_REF.supabase.co"
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_ROLE_KEY"
MISTRAL_API_KEY = "YOUR_MISTRAL_API_KEY"

try:
    supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Supabase terhubung!")
except Exception as e:
    supabase_client = None
    print("Supabase Error:", e)

print("Memuat model Semantic Search BAAI...")
model_bge = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Model Load OK!")

print("Memuat Evaluator (ROUGE & METEOR)...")
rouge_metric = evaluate.load('rouge')
meteor_metric = evaluate.load('meteor')
print("Evaluasi Load OK!")

### Step 1 & 2: Analisis Pertanyaan (Query Parsing) & Pencarian Paralel (Vector + Graph)

In [ ]:
# ---------- STEP 1: QUERY PARSING ---------- 
def extract_query_entities(query: str) -> list:
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {MISTRAL_API_KEY}"}
    prompt_msg = f"Extract ONLY the core medical or herbal keywords (Disease, Symptom, Plant) from this query: '{query}'. Return a valid JSON array of strings."
    try:
        resp = requests.post(url, headers=headers, json={
            "model": "mistral-small-latest",
            "messages": [{"role": "user", "content": prompt_msg}],
            "temperature": 0.1
        })
        resp.raise_for_status()
        content = resp.json()['choices'][0]['message']['content'].strip()
        clean_out = re.sub(r"^```json\n?", "", content)
        clean_out = re.sub(r"\n?```$", "", clean_out)
        parsed = json.loads(clean_out)
        if isinstance(parsed, list):
            return parsed
        return []
    except Exception as e:
        stopwords = ["what", "herb", "for", "is", "the", "a", "medical"]
        return [w for w in query.lower().split() if w not in stopwords]

# ---------- STEP 2: JALUR A (VECTOR RETRIEVAL) ---------- 
def retrieve_vector_chunks(query: str, match_function: str, limit: int = 4):
    query_vector = model_bge.encode([query], normalize_embeddings=True)[0].tolist()
    response = supabase_client.rpc(
        match_function,
        {
            "query_embedding": query_vector,
            "match_threshold": 0.3,
            "match_count": limit
        }
    ).execute()
    if not response.data:
        return []
    return response.data

# ---------- STEP 2: JALUR B (GRAPH RETRIEVAL) ---------- 
def retrieve_graph_relations(entities: list, table_name: str, limit: int = 10) -> list:
    relations = []
    for ent in entities:
        if len(ent) < 3: continue
        try:
            res = supabase_client.table(table_name).select('*').or_(f"entity_1.ilike.%{ent}%,entity_2.ilike.%{ent}%").limit(limit).execute()
            for row in res.data:
                relation_text = str(row['relation']).replace('_', ' ')
                relations.append(f"{row['entity_1']} {relation_text} {row['entity_2']}.")
        except Exception as e:
            pass
    return list(set(relations))


### Step 3 & 4: Hybrid Fusion Generation & Evaluator

In [ ]:
# ---------- STEP 3 & 4: HYBRID FUSION & GENERATION ---------- 
def ask_mistral_hybrid(query: str, vector_contexts: list, graph_contexts: list):
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {MISTRAL_API_KEY}"}
    
    # MENGOLAH DAN MEMBERSIHKAN VECTOR TEXT UNTUK FUSION
    vector_text_list = []
    for c in vector_contexts[:3]:  # AMBIL LEBIH BANYAK CHUNKS
        clean_text = c['text_content'].replace('\n', ' ')
        vector_text_list.append(f"- {clean_text}")
    vector_text = "\n".join(vector_text_list) if vector_text_list else "No further details."
    
    # MENGOLAH GRAPH TEXT
    graph_text = "\n- ".join(graph_contexts) if graph_contexts else "No relations found."
    if graph_contexts:
        graph_text = "- " + graph_text
    
    system_prompt = """
You are an AI tasked with generating a high-precision literature review summary.
Your objective is to maximize n-gram overlap (ROUGE/METEOR) with a human-written scientific baseline answer.
"""
    
    user_prompt = f"""
Question: {query}

=== [GRAPH RELATIONS] ===
{graph_text}

=== [VECTOR TEXT CHUNKS] ===
{vector_text}

INSTRUCTIONS FOR MAXIMUM ROUGE/METEOR:
1. HERB SELECTION: Look at the [VECTOR TEXT CHUNKS]. Find the MOST PROMINENT herb/plant discussed in relation to the disease. Use the [GRAPH RELATIONS] to verify the relationship, but ALWAYS prioritize the detailed herb in the Vector text.
2. DIRECT EXTRACTION (PLAGIARISM): Do NOT paraphrase creatively. You MUST extract sentences verbatim from the [VECTOR TEXT CHUNKS] that talk about the biological properties, studies, or components of that specific herb.
3. DISEASE CONTEXT: If the text says "[Disease] is a disease that is quite high in Indonesia..." or something similar, COPY IT EXACTLY. If it discusses "Several studies..." or "Researchers were therefore interested...", COPY IT EXACTLY.
4. Connect these verbatim extracted clauses smoothly into a single, dense paragraph (3-5 sentences). 
5. Exclude irrelevant nodes from the Graph that don't have rich details in the Vector text.
    """
    
    data = {
        "model": "mistral-small-latest",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1
    }
    
    resp = requests.post(url, headers=headers, json=data)
    resp.raise_for_status()
    return resp.json()['choices'][0]['message']['content']


# ---------- EVALUATOR PIPELINE ---------- 
def run_hybrid_evaluation(query: str, ground_truth: str):
    methods = [
        {"name": "Char Hybrid", "vector_rpc": "match_chunk_character", "graph_table": "graph_character"},
        {"name": "Word Hybrid", "vector_rpc": "match_chunk_word", "graph_table": "graph_word"},
        {"name": "Sent Hybrid", "vector_rpc": "match_chunk_sentence", "graph_table": "graph_sentence"}
    ]
    
    print(f"Q: {query}\n")
    
    # STEP 1: QUERY ENTITY EXTRACTION
    query_entities = extract_query_entities(query)
    print(f"[Query Parsing] Target Entities: {query_entities}\n")
    
    answers = {}
    
    for method in methods:
        print(f">>> {method['name']} Processing...")
        
        # STEP 2A: VECTOR RETRIEVAL
        vector_docs = retrieve_vector_chunks(query, method['vector_rpc'], limit=3)
        print(f"   [Jalur A Vector] Ditemukan {len(vector_docs)} chunks.")
        
        # STEP 2B: GRAPH RETRIEVAL
        graph_docs = retrieve_graph_relations(query_entities, method['graph_table'], limit=8)
        print(f"   [Jalur B Graph] Ditemukan {len(graph_docs)} relasi murni.")
        
        if not vector_docs and not graph_docs:
            ans = "I cannot answer based on the given context."
        else:
            try:
                # STEP 3 & 4: HYBRID FUSION GENERATION
                ans = ask_mistral_hybrid(query, vector_docs, graph_docs)
                ans = ans.replace('\n', ' ').strip(' ')
            except Exception as e:
                ans = f"[ERROR API MISTRAL] {e}"
                
        answers[method['name']] = ans
        print(f"   -> Final Answer: {ans[:200]}...\n")
        time.sleep(6) # Waktu tidur diperpanjang untuk mencegah rate-limiting
        
    print("\n=== HASIL METRIK ===")
    
    # 5. EVALUATION PROCESS (ROUGE & METEOR)
    for method in methods:
        ans = answers[method['name']]
        if ans.startswith("[ERROR") or "i cannot answer" in ans.lower():
            print(f"{method['name']}: ROUGE-1: 0.000, ROUGE-2: 0.000, ROUGE-L: 0.000 | METEOR: 0.000")
            continue
            
        try:
            rouge_res = rouge_metric.compute(predictions=[ans], references=[ground_truth])
            meteor_res = meteor_metric.compute(predictions=[ans], references=[ground_truth])
            
            r1 = rouge_res.get('rouge1', 0)
            r2 = rouge_res.get('rouge2', 0)
            rl = rouge_res.get('rougeL', 0)
            met = meteor_res.get('meteor', 0)
            
            print(f"{method['name']}: ROUGE-1: {r1:.3f}, ROUGE-2: {r2:.3f}, ROUGE-L: {rl:.3f} | METEOR: {met:.3f}")
        except Exception as e:
            print(f"{method['name']}: Error Scoring -> {e}")
            time.sleep(1)


---

### PENGUJIAN PERTANYAAN 1

In [ ]:
query1 = "Herb for headache"
gt1 = "Cinnamon is a spice that has anti-inflammatory and neuroprotective properties. Researchers were therefore interested in studying whether cinnamon could help reduce migraine attacks and inflammation. For example, this journal describes about that"

run_hybrid_evaluation(query1, gt1)

### PENGUJIAN PERTANYAAN 2

In [ ]:
query2 = "Herb For Diabetes"
gt2 = "Trigonella foenum-graecum is one of the important medicinal plants in the management of diabetes mellitus. Several studies, such as (Geberemeskel et al., 2019), have investigated the effect of Trigonella foenum-graecum seed powder on the lipid profile of newly diagnosed type II diabetic patients."

run_hybrid_evaluation(query2, gt2)

### PENGUJIAN PERTANYAAN 3

In [ ]:
query3 = "What Herb for hypertension"
gt3 = "Hypertension is a disease that is quite high in Indonesia, a major risk factor for cardiovascular disease (CVD). One of the herbs used is Centella asiatica (L) Urb. belongs to the Apiaceae (Umbelliferae) plant family, which has high triterpenoids and flavonoids and has antioxidant properties and is involved in the reninangiotensin-aldosterone system, which is an important hormonal system for blood pressure regulation."

run_hybrid_evaluation(query3, gt3)

### PENGUJIAN PERTANYAAN 4

In [ ]:
query4 = "Medical herb for fever"
gt4 = "that A. manihot and its bioactive constituents have a wide range of biological properties, including anti-diabetic nephropathy, antioxidant, antiadipogenic, anti-inflammatory, analgesic, anticonvulsant, antidepressant, antiviral, antitumor, cardioprotective, antiplatelet, neuroprotective activity, immunomodulatory, and hepatoprotective. And A.manihot can be used for fever. However, further studies and clinical trials are still needed to confirm these findings"

run_hybrid_evaluation(query4, gt4)

### PENGUJIAN PERTANYAAN 5

In [ ]:
query5 = "Medical herb for rheumatism"
gt5 = "Rheumatoid arthritis is one part of rheumatic disease. In Indonesia, some herbs that are often used for rheumatism are jambe jackfruit, and several other examples, such as cinnamon, curcumin, African tree, and so on."

run_hybrid_evaluation(query5, gt5)

### PENGUJIAN PERTANYAAN 6

In [ ]:
query6 = "Medical herb for Heartburn"
gt6 = "Some sources have been researched, such as Harvard Medical School, which states that ginger root is a popular herbal remedy for heartburn. It has been used for centuries to relieve the symptoms of heartburn, such as a burning sensation in the chest"

run_hybrid_evaluation(query6, gt6)